# 04 · 预测模型与可解释性

## 预测设定

用 **t-1 期**的宏观特征预测 **t 期**的**生效评级**（`score_in_effect`，1-21）。
使用滞后特征而非同期特征，是避免伪回归的关键：评级机构在 t 年初看到的只是 t-1 年的宏观数据。

## 模型

| 模型 | 角色 | 优点 | 局限 |
| --- | --- | --- | --- |
| 有序 Logit / Probit | 基准 | 尊重评级的**有序性**、参数可解释 | 平行回归线假设 |
| 随机森林 | 非线性基准 | 自动捕捉交互与非线性 | 对不平衡敏感、外推能力弱 |
| XGBoost | 性能上限 | 最强的表格数据表现 | 需调参、可解释性依赖 SHAP |

## 评估指标（为何不只看准确率）

评级分布极度不平衡，全部预测众数档也能得到不低的准确率。因此必须同时报告：

* `balanced_accuracy` / `f1_macro`：对少数档位给予同等权重
* `adjacent_acc_1` / `adjacent_acc_2`：**相差 1 档 / 2 档以内**的比例（实务价值最高）
* `mae_ordinal`：档位距离的平均绝对误差
* `p_downgrade`：预测分布中低于「预测前评级」的概率之和——即**下调预警概率**

## 交叉验证

按年**前向链式**（expanding window）：训练集严格早于测试年份。
随机 K 折会让模型「看到未来」，产生严重乐观偏差。

> ⚠️ 默认使用合成演示数据，结果不构成实证结论。

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name and not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import get_path
from src.features.macro_features import (
    DEFAULT_FEATURES,
    add_derived_features,
    add_lags,
    build_feature_matrix,
)
from src.models.evaluate import classification_metrics, confusion_table, feature_importance_table
from src.models.ordinal import OrdinalProbabilityModel, fit_ordinal_logit, fit_ordinal_probit
from src.models.timeseries_cv import compare_cv_results, cross_validate_model, expanding_year_folds
from src.models.tree_models import (
    OrdinalTreeClassifier,
    build_random_forest,
    fit_tree_model,
)
from src.pipeline import attach_migration_probabilities

PANEL_PATH = get_path("processed") / "panel_country_year.csv"
if PANEL_PATH.exists():
    panel = pd.read_csv(PANEL_PATH, encoding="utf-8-sig")
else:
    from src.clean.panel import build_country_year_panel
    from src.ingest.ratings import load_sample_ratings

    panel = build_country_year_panel(
        load_sample_ratings(),
        pd.read_csv(get_path("sample") / "macro_sample.csv", encoding="utf-8-sig"),
        start_year=2000,
        end_year=2023,
    )
panel = add_derived_features(panel, group_cols=("country_iso3", "agency"))
FEATURES = [c for c in DEFAULT_FEATURES if c in panel.columns]
print(f"面板 {panel.shape}，特征 {len(FEATURES)} 个")

## 1. 构造建模矩阵（滞后一期特征）

In [ ]:
ENTITY_GROUP = ("country_iso3", "agency")

lagged = add_lags(panel, FEATURES, group_cols=ENTITY_GROUP, time_col="year", lags=(1,))
lagged = lagged.sort_values([*ENTITY_GROUP, "year"])
lagged["prior_score"] = lagged.groupby(list(ENTITY_GROUP), dropna=False)["score_in_effect"].shift(1)

LAG_COLS = [f"{c}_lag1" for c in FEATURES]
X, y, meta = build_feature_matrix(lagged, LAG_COLS, target="score_in_effect")
years = meta["year"].to_numpy()
prior_scores = lagged.loc[X.index, "prior_score"].to_numpy(dtype="float64")
print(f"建模矩阵: {X.shape[0]:,} 行 × {X.shape[1]} 特征 | 目标取值数: {y.nunique()}")
print(f"年份范围: {years.min()}-{years.max()}")
y.value_counts().sort_index().to_frame("样本数")

## 2. 时间序列交叉验证折

In [ ]:
folds = expanding_year_folds(pd.Series(years), n_splits=5, min_train_years=10)
for fold in folds:
    print(
        f"fold{fold.fold}: 训练 {min(fold.train_years)}-{max(fold.train_years)} "
        f"(n={len(fold.train_index)}) → 测试 {fold.test_years} (n={len(fold.test_index)})"
    )

## 3. 多模型交叉验证

> 若样本很小，下面的单元可能需要几分钟。可把 `CV_KWARGS` 中的 `n_splits` 调小。

In [ ]:
CV_KWARGS = {"n_splits": 5, "min_train_years": 10, "scheme": "expanding"}
outputs_cv = []

outputs_cv.append(
    cross_validate_model(lambda: OrdinalProbabilityModel("logit"), X, y, years,
                         model_name="ordinal_logit", **CV_KWARGS)
)
outputs_cv.append(
    cross_validate_model(lambda: OrdinalProbabilityModel("probit"), X, y, years,
                         model_name="ordinal_probit", **CV_KWARGS)
)
outputs_cv.append(
    cross_validate_model(lambda: OrdinalTreeClassifier(backend="random_forest"), X, y, years,
                         model_name="random_forest", **CV_KWARGS)
)
try:
    outputs_cv.append(
        cross_validate_model(lambda: OrdinalTreeClassifier(backend="xgboost"), X, y, years,
                             model_name="xgboost", **CV_KWARGS)
    )
except Exception as exc:
    print(f"XGBoost 跳过: {exc}")

In [ ]:
comparison = compare_cv_results(outputs_cv)
pivot = comparison.pivot(index="metric", columns="model", values="mean")
pivot.round(4)

In [ ]:
for output in outputs_cv:
    print(f"\n===== {output['model']} 逐折指标 =====")
    display(output["fold_metrics"][["fold", "test_years", "test_n", "accuracy", "adjacent_acc_1", "mae_ordinal"]].round(4))

## 4. 下调预警概率（样本外）

把有序模型的概率分布按「预测前评级」拆分：
**下调概率** = 预测分布中低于预测前评级的所有档位概率之和。

In [ ]:
best = outputs_cv[0]  # 有序 Logit
prob_cols = [str(c) for c in best["classes"]]
oof = pd.DataFrame({
    "country_iso3": meta["country_iso3"].to_numpy(),
    "year": years,
    "prior_score": prior_scores,
    "y_true": y.to_numpy(),
    "y_pred": best["oof_pred"].to_numpy(),
})
proba = best["oof_proba"].copy()
proba.columns = prob_cols
oof = pd.concat([oof, proba[[c for c in prob_cols if c in proba.columns]]], axis=1)
oof = attach_migration_probabilities(oof, [c for c in prob_cols if c in proba.columns], prior_scores)

print("样本外下调概率的分布：")
display(oof["p_downgrade"].describe().round(4))

# 实际下调样本的预警概率是否更高？（简易判别力检查）
actual_down = (oof["y_true"] < oof["prior_score"]).astype(int)
print("\n实际发生下调 vs 未下调的平均预警概率：")
print(oof.groupby(actual_down)["p_downgrade"].mean().round(4))

## 5. 全样本拟合、特征重要性与 SHAP

In [ ]:
ordinal = fit_ordinal_logit(X, y)
print(ordinal.summary_text())
print(f"\nMcFadden 伪 R²: {ordinal.prsquared:.4f}")

In [ ]:
rf = fit_tree_model(build_random_forest(), X, y)
importance = feature_importance_table(rf, list(X.columns), top_n=15)
importance.round(4)

In [ ]:
from src.models.explain import explain_with_shap, shap_available

print("SHAP 可用:", shap_available())
shap_result = explain_with_shap(rf, X, max_samples=400)
if shap_result.ok:
    shap_summary = shap_result.to_frames()["summary"]
    display(shap_summary.round(5))
else:
    print("SHAP 未产出结果：", shap_result.message)
    print("可改用内置特征重要性（上一单元）作为替代。")

In [ ]:
if shap_result.ok:
    from src.visualization.interactive import shap_bar_plotly

    shap_bar_plotly(shap_result.to_frames()["summary"], top_n=15)

In [ ]:
predictions = np.asarray(rf.predict(X))
confusion = confusion_table(y, predictions, labels=sorted(pd.unique(y)))
display(confusion)
display(pd.Series(classification_metrics(y, predictions, labels=sorted(pd.unique(y)))).round(4).to_frame("训练集"))